# Multilingual Health QA Notebook 1: EDA + Baseline
**Competition:** Multilingual Health Question Answering in Low-Resource African Languages (Zindi)    
**Experiment:** Baseline (zero-shot mt5-small ) with simple prompt  

---
## Experiment Log
|  | Description | ROUGE-1 | ROUGE-L | Zindi Score |
|---|---|---|---|---|
| 1 | Baseline: zero-shot mt5-small | 0.0028 | 0.0028 | 0.000926 |


## 0. Setup — Install Libraries

In [ ]:
# Install required libraries
!pip install transformers datasets evaluate rouge_score sentencepiece -q
print('Libraries installed successfully!')

In [ ]:
# Core imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# HuggingFace imports
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Check GPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU detected. Go to Runtime > Change runtime type > T4 GPU')

## 1. Mount Google Drive & Load Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ===================================================
# UPDATE THIS PATH to where you uploaded your data
# ===================================================
DATA_PATH = '/content/drive/MyDrive/health_qa_data/'

train_df = pd.read_csv(DATA_PATH + 'Train.csv')
val_df   = pd.read_csv(DATA_PATH + 'Val.csv')
test_df  = pd.read_csv(DATA_PATH + 'Test.csv')
sample_df = pd.read_csv(DATA_PATH + 'SampleSubmission.csv')

print(f'Train: {len(train_df):,} rows')
print(f'Val:   {len(val_df):,} rows')
print(f'Test:  {len(test_df):,} rows')
print()
print('Train columns:', train_df.columns.tolist())
print('Test columns:', test_df.columns.tolist())
print('Submission columns:', sample_df.columns.tolist())

## 2. Exploratory Data Analysis (EDA)

In [ ]:
# Preview training data
print('=== TRAIN SAMPLE ===')
train_df.head(3)

In [ ]:
# Check for missing values
print('=== MISSING VALUES ===')
print('Train:')
print(train_df.isnull().sum())
print()
print('Val:')
print(val_df.isnull().sum())
print()
print('Test:')
print(test_df.isnull().sum())

In [ ]:
# Language/subset distribution
subset_counts = train_df['subset'].value_counts()
print('=== LANGUAGE DISTRIBUTION (TRAIN) ===')
print(subset_counts)
print()

# Language name mapping
lang_map = {
    'Eng_Uga': 'English (Uganda)',
    'Eng_Gha': 'English (Ghana)',
    'Aka_Gha': 'Akan (Ghana)',
    'Lug_Uga': 'Luganda (Uganda)',
    'Eng_Eth': 'English (Ethiopia)',
    'Swa_Ken': 'Kiswahili (Kenya)',
    'Eng_Ken': 'English (Kenya)',
    'Amh_Eth': 'Amharic (Ethiopia)'
}
train_df['language'] = train_df['subset'].map(lang_map)
val_df['language']   = val_df['subset'].map(lang_map)
test_df['language']  = test_df['subset'].map(lang_map)

In [ ]:
# Plot language distribution
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Train distribution
train_counts = train_df['language'].value_counts()
axes[0].barh(train_counts.index, train_counts.values, color='steelblue')
axes[0].set_title('Train Set — Language Distribution', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Number of Examples')
for i, v in enumerate(train_counts.values):
    axes[0].text(v + 50, i, str(v), va='center')

# Test distribution
test_counts = test_df['language'].value_counts()
axes[1].barh(test_counts.index, test_counts.values, color='coral')
axes[1].set_title('Test Set — Language Distribution', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Number of Examples')
for i, v in enumerate(test_counts.values):
    axes[1].text(v + 5, i, str(v), va='center')

plt.tight_layout()
plt.savefig('/content/language_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved.')

In [ ]:
# Text length analysis
train_df['input_len']  = train_df['input'].str.len()
train_df['output_len'] = train_df['output'].str.len()
train_df['input_words']  = train_df['input'].str.split().str.len()
train_df['output_words'] = train_df['output'].str.split().str.len()

print('=== INPUT LENGTH (characters) ===')
print(train_df['input_len'].describe().round(1))
print()
print('=== OUTPUT LENGTH (characters) ===')
print(train_df['output_len'].describe().round(1))
print()
print('=== INPUT LENGTH (words) ===')
print(train_df['input_words'].describe().round(1))
print()
print('=== OUTPUT LENGTH (words) ===')
print(train_df['output_words'].describe().round(1))

In [ ]:
# Plot length distributions
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

axes[0][0].hist(train_df['input_words'], bins=50, color='steelblue', edgecolor='white')
axes[0][0].set_title('Input Word Count Distribution')
axes[0][0].set_xlabel('Words')
axes[0][0].set_ylabel('Count')

axes[0][1].hist(train_df['output_words'], bins=50, color='coral', edgecolor='white')
axes[0][1].set_title('Output Word Count Distribution')
axes[0][1].set_xlabel('Words')

# Length by language
lang_input_len = train_df.groupby('language')['input_words'].mean().sort_values()
axes[1][0].barh(lang_input_len.index, lang_input_len.values, color='steelblue')
axes[1][0].set_title('Avg Input Length by Language (words)')

lang_output_len = train_df.groupby('language')['output_words'].mean().sort_values()
axes[1][1].barh(lang_output_len.index, lang_output_len.values, color='coral')
axes[1][1].set_title('Avg Output Length by Language (words)')

plt.tight_layout()
plt.savefig('/content/length_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Show sample Q&A pairs per language
print('=== SAMPLE Q&A PAIRS PER LANGUAGE ===')
for subset in train_df['subset'].unique():
    sample = train_df[train_df['subset'] == subset].iloc[0]
    print(f'\n--- {lang_map[subset]} ({subset}) ---')
    print(f'Q: {sample["input"][:200]}...')
    print(f'A: {sample["output"][:200]}...')

In [ ]:
# EDA Summary
print('=== EDA SUMMARY ===')
print(f'Total training examples: {len(train_df):,}')
print(f'Total languages: {train_df["subset"].nunique()}')
print(f'Average question length: {train_df["input_words"].mean():.1f} words')
print(f'Average answer length: {train_df["output_words"].mean():.1f} words')
print(f'Max question length: {train_df["input_words"].max()} words')
print(f'Max answer length: {train_df["output_words"].max()} words')
print(f'Missing values in train input: {train_df["input"].isnull().sum()}')
print(f'Missing values in train output: {train_df["output"].isnull().sum()}')

print('\nClass imbalance: Eng_Uga has 7624 samples vs Amh_Eth with only 1845.')
print('This is a key challenge — low-resource languages need special attention.')

## 3. Preprocessing

**Decisions made:**
1. Strip whitespace from inputs/outputs
2. Drop rows with null inputs or outputs
3. Add a language prefix to the input (e.g., `translate Akan: <question>`) — helps the model understand language context
4. Cap token length at 512 for input and 256 for output (based on EDA)


In [ ]:
# Preprocessing function
def preprocess_df(df, is_test=False):
    df = df.copy()
    
    # Strip whitespace
    df['input'] = df['input'].str.strip()
    if not is_test:
        df['output'] = df['output'].str.strip()
    
    # Drop nulls
    before = len(df)
    df = df.dropna(subset=['input'])
    if not is_test:
        df = df.dropna(subset=['output'])
    after = len(df)
    if before != after:
        print(f'  Dropped {before - after} rows with null values')
    
    # Add language prefix to input
    # This helps the model understand what language context to use
    subset_to_prefix = {
        'Aka_Gha':  'answer in Akan: ',
        'Amh_Eth':  'answer in Amharic: ',
        'Eng_Eth':  'answer in English: ',
        'Eng_Gha':  'answer in English: ',
        'Eng_Ken':  'answer in English: ',
        'Eng_Uga':  'answer in English: ',
        'Lug_Uga':  'answer in Luganda: ',
        'Swa_Ken':  'answer in Kiswahili: ',
    }
    df['model_input'] = df['subset'].map(subset_to_prefix) + df['input']
    
    return df

print('Preprocessing train...')
train_clean = preprocess_df(train_df)
print(f'  Train: {len(train_clean):,} rows')

print('Preprocessing val...')
val_clean = preprocess_df(val_df)
print(f'  Val: {len(val_clean):,} rows')

print('Preprocessing test...')
test_clean = preprocess_df(test_df, is_test=True)
print(f'  Test: {len(test_clean):,} rows')

print()
print('Sample preprocessed input:')
print(train_clean['model_input'].iloc[0][:300])

## 4. Experiment 1 Baseline: Zero-shot mt5-small

mt5-small is pretrained on multilingual data including some African languages. Without any fine-tuning, it may produce some reasonable answers. This gives us a true baseline score to compare all future experiments against.

In [ ]:
# Load mt5-small tokenizer and model (zero-shot — no fine-tuning yet)
MODEL_NAME = 'google/mt5-small'

print(f'Loading {MODEL_NAME}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
model = model.to(device)
print(f'Model loaded! Parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# Generation function
def generate_answers(texts, model, tokenizer, batch_size=16,
                     max_input_length=512, max_output_length=256):
    """
    Generate answers for a list of input texts in batches.
    Returns a list of generated strings.
    """
    model.eval()
    all_outputs = []
    
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        
        inputs = tokenizer(
            batch,
            return_tensors='pt',
            max_length=max_input_length,
            truncation=True,
            padding=True
        ).to(device)
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_output_length,
                num_beams=4,
                early_stopping=True,
                no_repeat_ngram_size=3
            )
        
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        all_outputs.extend(decoded)
        
        if (i // batch_size) % 10 == 0:
            print(f'  Processed {i+len(batch)}/{len(texts)} examples...')
    
    return all_outputs

print('Generation function defined.')

In [ ]:
# Quick test on 5 examples before running the full set
print('=== QUICK TEST (5 examples) ===')
sample_inputs = val_clean['model_input'].tolist()[:5]
sample_refs   = val_clean['output'].tolist()[:5]

sample_preds = generate_answers(sample_inputs, model, tokenizer, batch_size=5)

for i, (inp, ref, pred) in enumerate(zip(sample_inputs, sample_refs, sample_preds)):
    print(f'\n--- Example {i+1} ---')
    print(f'Input:     {inp[:150]}...')
    print(f'Reference: {ref[:150]}...')
    print(f'Predicted: {pred[:150]}...')

In [ ]:
# Evaluate on validation set using ROUGE
import evaluate
rouge = evaluate.load('rouge')

def compute_rouge(predictions, references):
    results = rouge.compute(
        predictions=predictions,
        references=references,
        use_stemmer=False
    )
    return {
        'rouge1': round(results['rouge1'], 4),
        'rougeL': round(results['rougeL'], 4)
    }

print('Evaluating on validation set (this may take a few minutes)...')
# Use a subset for speed — first 500 examples
val_inputs = val_clean['model_input'].tolist()[:500]
val_refs   = val_clean['output'].tolist()[:500]

val_preds = generate_answers(val_inputs, model, tokenizer, batch_size=16)

scores = compute_rouge(val_preds, val_refs)
print(f'\n=== EXPERIMENT 1 RESULTS (Zero-shot mt5-small) ===')
print(f'ROUGE-1: {scores["rouge1"]}')
print(f'ROUGE-L: {scores["rougeL"]}')
print()
print('NOTE: Zero-shot performance is expected to be low.')
print('This is the baseline we will improve through fine-tuning.')

In [ ]:
# Per-language ROUGE scores
print('=== PER-LANGUAGE ROUGE SCORES (val subset, 500 examples) ===')
val_subset_500 = val_clean.iloc[:500]

per_lang_results = {}
for subset in val_subset_500['subset'].unique():
    mask = val_subset_500['subset'] == subset
    idxs = mask[mask].index.tolist()
    # Map back to position in val_preds
    positions = [val_clean.iloc[:500].index.tolist().index(i) for i in idxs if i in val_clean.iloc[:500].index.tolist()]
    if not positions:
        continue
    preds_lang = [val_preds[p] for p in positions]
    refs_lang  = [val_refs[p]  for p in positions]
    if preds_lang:
        lang_scores = compute_rouge(preds_lang, refs_lang)
        per_lang_results[lang_map.get(subset, subset)] = lang_scores
        print(f'{lang_map.get(subset, subset):30s} ROUGE-1: {lang_scores["rouge1"]:.4f}  ROUGE-L: {lang_scores["rougeL"]:.4f}')

## 5. Generate Test Predictions & Create Submission

In [ ]:
# Generate predictions for ALL test examples
print(f'Generating predictions for {len(test_clean)} test examples...')
print('This will take ~10-15 minutes on free Colab...')

test_inputs = test_clean['model_input'].tolist()
test_preds  = generate_answers(test_inputs, model, tokenizer, batch_size=16)

print(f'Done! Generated {len(test_preds)} predictions.')

In [ ]:
submission = pd.DataFrame({
    'ID':         test_clean['ID'].tolist(),
    'TargetRLF1': test_preds,
    'TargetR1F1': test_preds,
    'TargetLLM':  test_preds
})

# Verify format matches sample submission
print('=== SUBMISSION PREVIEW ===')
print(submission.head())
print(f'\nShape: {submission.shape}')
print(f'Columns: {submission.columns.tolist()}')
assert len(submission) == 2618, f'Expected 2618 rows, got {len(submission)}'
assert list(submission.columns) == ['ID', 'TargetRLF1', 'TargetR1F1', 'TargetLLM']
print('\nFormat check PASSED!')

In [ ]:
# Save submission
SUBMISSION_PATH = '/content/drive/MyDrive/health_qa_data/submissions/'
import os
os.makedirs(SUBMISSION_PATH, exist_ok=True)

submission_file = SUBMISSION_PATH + 'submission_exp1_zeroshot_mt5small.csv'
submission.to_csv(submission_file, index=False)
print(f'Submission saved to: {submission_file}')
print()
print('NEXT STEPS:')
print('1. Download this CSV from Google Drive')
print('2. Go to Zindi competition page')
print('3. Click Submit and upload the file')
print('4. Screenshot your leaderboard score and add it to your experiment log above')

## 6. Experiment Log & Insights

After submitting, fill in the table below:

| Experiment | Model | Preprocessing | Val ROUGE-1 | Val ROUGE-L | Leaderboard Score | Key Insight |
|---|---|---|---|---|---|---|
| 1 | mt5-small (zero-shot) | Language prefix | 0.0028 | 0.0028 | 0.000926 | Zeroshot baseline model has no task specific training |

**Observations from EDA:**
- The dataset is imbalanced: English (Uganda) dominates with 7,624 examples; Amharic (Ethiopia) has only 1,845
- Average answer length (approximately 65 words) is much longer than the question (20 words) this is a generation task, not extractive
- 5 distinct language families are represented (English, Akan, Luganda, Kiswahili, Amharic) a single multilingual model must handle all of them
- No missing values found in training data (clean dataset)